# P14′: Qwen2.5-Omni-3B zero-shot + CONSTRUCT, **protocol chính thức** của OV-MER (so được trực tiếp với SOTA). Kaggle **GPU T4 ×2**
Chạy notebook này **3 lần**, mỗi lần đổi `PART` (một tập dữ liệu mỗi lần, mỗi lần dưới 8 giờ):
| PART | Tập | Số clip | Thời gian dự kiến |
|---|---|---|---|
| 1 | OV-MERD+ (benchmark có số SOTA) | 532 | ~3,5–4 giờ |
| 2 | MER2025-FG | 1.000 | ~6,5–7 giờ |
| 3 | EmoS-MELD mới | 400 | ~3 giờ |
Secret: **HF_TOKEN** (và **MER2026_ZIP_PW** nếu được báo thiếu). Accelerator **GPU T4 x2**, Internet **On**. Thiết kế và tiêu chí: `explore/ovmer/p14o_score.py`.

In [ ]:
PART = 1   # 1 = OV-MERD+, 2 = MER2025-FG, 3 = EmoS-MELD-fresh
DS = {1: 'OV-MERD+', 2: 'MER2025-FG', 3: 'EmoS-MELD-fresh'}[PART]
MODEL, NSAMP = 'Qwen/Qwen2.5-Omni-3B', 16
KOUT, WORK = '/kaggle/working', '/tmp/p14'
MELD_TAR = 'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz'

In [ ]:
import os, subprocess
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
os.makedirs(WORK, exist_ok=True)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
sh(f'rm -rf {WORK}/affectgpt && git clone -q --depth 1 https://github.com/zeroQiaoba/AffectGPT {WORK}/affectgpt')
sh(f'rm -rf {WORK}/emos && git clone -q --depth 1 https://github.com/NLP2CT/EmoS {WORK}/emos')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub qwen-omni-utils openpyxl' + (' bitsandbytes'), check=False)
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise SystemExit('HF_TOKEN secret is not attached to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN, then Save & Run All again.') from None
try:
    os.environ['MER2026_ZIP_PW'] = UserSecretsClient().get_secret('MER2026_ZIP_PW')   # only needed if the zip is encrypted; never printed
except Exception:
    pass
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch, transformers; print('GPUs:', torch.cuda.device_count(), '| transformers', transformers.__version__)

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
Z26 = hf_hub_download(repo_id='MERChallenge/MER2026', repo_type='dataset', filename='video_7z/video_track2_train_human/video_split.zip',
                      local_dir=f'{WORK}/MER2026', token=os.environ['HF_TOKEN'])
print('MER2026 human video zip', f'{os.path.getsize(Z26) / 1e9:.2f} GB')
snapshot_download(MODEL); snapshot_download('Qwen/Qwen2.5-7B-Instruct')  # download once, before the two worker processes start
ITEMS = f'{KOUT}/p14_items.jsonl'
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_prepare.py', '--p14', '--affectgpt', f'{WORK}/affectgpt', '--emos', f'{WORK}/emos',
                '--mer26zip', Z26, '--meld_tar', MELD_TAR, '--work', f'{WORK}/media', '--out', ITEMS], check=False)

In [ ]:
import time
def run_parallel(make_cmd, logname):
    procs = []
    for g in range(2):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append(subprocess.Popen(make_cmd(g), env=env, stdout=open(f'{KOUT}/{logname}_{g}.txt', 'w'), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        for g in range(2):
            lines = [l for l in open(f'{KOUT}/{logname}_{g}.txt', errors='ignore') if 'clips,' in l or 'extracted' in l or 'fail' in l or 'Error' in l]
            print(f'{logname} GPU{g}:', lines[-1].strip() if lines else '(starting)', flush=True)
    print(logname, 'return codes:', [p.returncode for p in procs])
R = f'{WORK}/icme/explore/ovmer/p13_mbr.py'; O = f'{WORK}/icme/explore/ovmer/p14_official.py'
run_parallel(lambda g: ['python', R, 'run', '--items', ITEMS, '--out', f'{KOUT}/desc_{PART}_{g}.jsonl', '--hours', '7', '--model', MODEL,
                        '--n_samples', str(NSAMP), '--shard', f'{g}/2', '--protocol', 'official', '--only_ds', DS], 'desc')
DESC = f'{KOUT}/desc_{PART}.jsonl'
with open(DESC, 'w') as f:
    for g in range(2):
        if os.path.exists(f'{KOUT}/desc_{PART}_{g}.jsonl'): f.write(open(f'{KOUT}/desc_{PART}_{g}.jsonl').read())
run_parallel(lambda g: ['python', O, 'extract', '--descs', DESC, '--out', f'{KOUT}/labels_{PART}_{g}.jsonl', '--shard', f'{g}/2'], 'extract')
LAB = f'{KOUT}/p14_labels_part{PART}.jsonl'
with open(LAB, 'w') as f:
    for g in range(2):
        if os.path.exists(f'{KOUT}/labels_{PART}_{g}.jsonl'): f.write(open(f'{KOUT}/labels_{PART}_{g}.jsonl').read())
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p14o_score.py', '--items', ITEMS, '--labels', LAB,
                '--mapping', f'{WORK}/affectgpt/AffectGPT-RL/emotion_wheel/wheel_mapping.npz', '--report', f'{KOUT}/p14_report_part{PART}.md'], check=False)
print(open(f'{KOUT}/p14_report_part{PART}.md').read())